# S2 — Faciès pétroliers et validation par puits

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/inphb/ic1/notebooks/IC1_S2_facies_par_puits.ipynb)
> Données : dossier `IC1_TP_donnees`. Bibliothèques : pandas, numpy, matplotlib, scikit-learn.

In [ ]:
# Configuration : Colab ou Jupyter local. Exécutez cette cellule en premier.
# Elle récupère les données et les tests de vérification : depuis le dépôt
# (../data, ../tests) si vous l'avez téléchargé, sinon depuis GitHub.
import os, shutil, urllib.request
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/inphb/ic1/'
FICHIERS = [('data/facies_diagraphies.csv', 'facies_diagraphies.csv'), ('tests/s2/q1.py', 'tests/q1.py'), ('tests/s2/q2.py', 'tests/q2.py'), ('tests/s2/q3.py', 'tests/q3.py')]
os.makedirs('tests', exist_ok=True)
for source, cible in FICHIERS:
    if os.path.exists(cible):
        continue
    local = os.path.join('..', source)
    if os.path.exists(local):
        shutil.copy(local, cible)                      # kit hors ligne / dépôt cloné
    else:
        urllib.request.urlretrieve(URL + source, cible)  # Colab : téléchargement
print('Prêt :', ', '.join(c for _, c in FICHIERS))

In [ ]:
# Vérification automatique des réponses (Otter-Grader)
try:
    import otter
except ImportError:
    %pip install -q otter-grader
    import otter
grader = otter.Notebook(tests_dir="tests")

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, StratifiedKFold, GroupKFold, cross_val_score, cross_val_predict, learning_curve, GridSearchCV
from sklearn.metrics import mean_absolute_error, r2_score, roc_auc_score, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
fa = pd.read_csv('facies_diagraphies.csv')
X = fa[['GR_api', 'RHOB_gcc', 'NPHI_vv', 'logRT_ohmm', 'PE_be']]; y, puits = fa['facies'], fa['puits']

## 1. k-fold aléatoire vs GroupKFold par puits

In [ ]:
modeles = {'knn': make_pipeline(StandardScaler(), KNeighborsClassifier(15)),
 'logit': make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
 'rf': make_pipeline(StandardScaler(), RandomForestClassifier(300, min_samples_leaf=3, random_state=0))}
scores = {}
for nom, m in modeles.items():
    a = cross_val_score(m, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=0)).mean()
    b = cross_val_score(m, X, y, cv=GroupKFold(6), groups=puits).mean()
    scores[nom] = (a, b)
    print(nom, round(a, 3), round(b, 3))

## 2. Matrice de confusion (puits jamais vus)

In [ ]:
pred = cross_val_predict(modeles['rf'], X, y, cv=GroupKFold(6), groups=puits)
ConfusionMatrixDisplay.from_predictions(y, pred, normalize='true', values_format='.2f'); plt.show()

## Exercices vérifiés automatiquement
Remplacez chaque `...` par votre réponse, exécutez la cellule : la vérification indique tout de suite si c'est correct.

**Q1.** Pour la forêt (`'rf'`), de combien l'exactitude en k-fold aléatoire dépasse-t-elle celle de la validation par puits ? Utilisez le dictionnaire `scores` (`scores[nom] = (aléatoire, par puits)`) et stockez l'écart dans `ecart_rf`.
Pourquoi le k-fold aléatoire est-il trop optimiste ici ?

In [ ]:
ecart_rf = ...
grader.check("q1")

**Q2.** D'après la matrice de confusion (puits jamais vus), quel faciès est le moins bien reconnu (plus petit rappel) ? Stockez son nom exact dans `facies_plus_confondu`.

In [ ]:
facies_plus_confondu = ...
grader.check("q2")

**Q3 (bonus).** Normalisez GR **par puits** (`fa.groupby('puits')['GR_api'].transform(lambda s: (s - s.mean()) / s.std())`), remplacez la colonne `GR_api` d'une copie de `X`, puis refaites la validation `GroupKFold(6)` avec `modeles['rf']`. Stockez l'exactitude moyenne dans `acc_gr_norm`.
 La normalisation aide-t-elle ? Pourquoi ?

In [ ]:
acc_gr_norm = ...
grader.check("q3")

Pour tout vérifier d'un coup :

In [ ]:
grader.check_all()